In [1]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import silhouette_score
from IPython.display import HTML

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
DOC_THEMES_DIR = DATA_DIR / 'THEMES_RESULTS_V4' / 'docs_with_themes.csv'
THEMES_DIR = DATA_DIR / 'THEMES_RESULTS_V4' / 'themes.csv'
Themes_aug_DIR = DATA_DIR / 'Themes_augmented_3.csv'


In [4]:
# Attach Ollama-generated theme labels without overwriting originals
themes_df = pd.read_csv(THEMES_DIR)
ollama_labels = (
    pd.read_csv(Themes_aug_DIR)
    .rename(columns={"caption_id": "theme_id", "gen_label": "llm_theme_label"})
)
ollama_themes_df = themes_df.merge(ollama_labels, on="theme_id", how="left")
ollama_themes_df.head(10)

## Analyzing funiness over time

In [3]:
# Load themes
themes_df = pd.read_csv(THEMES_DIR)

# Sort by mean funniness and select top 10
top10_funny_themes = (
    themes_df
    .sort_values("mu_mean", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Display / inspect
top10_funny_themes[
    ["theme_id", "theme_label", "mu_mean", "nb_instances"]
]


,theme_id,theme_label,mu_mean,nb_instances
0,2,Year,1.226651,21754
1,22,Know Feel,1.226381,21192
2,51,Evolution,1.225645,21676
3,58,Lawyer,1.225457,27958
4,39,Insurance Cover,1.225239,20720
5,63,Gentrification,1.224371,20270
6,20,Wear Mask,1.223832,21909
7,49,Tell,1.223448,37933
8,45,Marriage,1.223011,35378
9,44,Look Ant,1.222959,21734


In [6]:
top10_funny_themes_llm = (
    themes_df
    .sort_values("mu_mean", ascending=False)
    .head(10)
    .loc[:, ["theme_id", "theme_label_llm", "mu_mean", "nb_instances"]]
    .rename(columns={"theme_label_llm": "theme_label"})
)

top10_funny_themes_llm



,theme_id,theme_label,mu_mean,nb_instances
2,2,Time,1.226651,21754
22,22,Sensation,1.226381,21192
51,51,Evolution,1.225645,21676
58,58,Justice,1.225457,27958
39,39,Insurance,1.225239,20720
63,63,Housing,1.224371,20270
20,20,Covid-19,1.223832,21909
49,49,Communication,1.223448,37933
45,45,Marriage,1.223011,35378
44,44,Observe,1.222959,21734


I'm going to work with Olama generated themes

In [ ]:
# Add contest publish year info to docs_with_themes
# Contest 510 (Feb 22, 2016) is used as the anchor publish date and contests occur weekly.
docs_with_themes = pd.read_csv(DOC_THEMES_DIR)
contest_zero_date = pd.Timestamp('2016-02-22')
contest_start_id = 510

contest_dates = (
    docs_with_themes[['contest_id']]
    .drop_duplicates()
    .assign(
        contest_day_publish=lambda df:
            contest_zero_date + pd.to_timedelta(
                (df['contest_id'].astype(int) - contest_start_id) * 7,
                unit='D'
            )
    )
)
contest_dates['contest_year'] = contest_dates['contest_day_publish'].dt.year

docs_with_year = docs_with_themes.merge(contest_dates, on='contest_id', how='left')
docs_with_year[['contest_id', 'contest_day_publish', 'contest_year']].head(10)
